# C1 experiment log

Track C, Apetit Climate & Nature Navigator 2026.

`c1_weather_features.ipynb` is the finished pipeline. This notebook records the experiments that shaped it, in the order we ran them, and what each one changed.

Experiments 1 and 2 ran on older versions of the code. Those versions are kept below as the `legacy_*` functions, so the results here are the ones we saw at the time.

| # | Question | What we found |
|---|---|---|
| 1 | First tuning run: which degree-days to harvest and sowing temperature fit best? Both pea regions scored together, sensitive periods fixed | Clear best values for turnip rape and potato. Peas showed only a shallow dip. Spinach and carrot gave wildly wrong results at a few settings |
| 2 | Why did spinach and carrot go wrong? | Tiny rounding errors in one measure got a huge coefficient. Fixed in `cropweather.py` and `yieldcv.py` |
| 3 | Can Finnish and Swedish peas share one calendar? | No. Finland fits best at about 1,150 degree-days and Sweden at about 600, so they're tuned separately |
| 4 | What do the pea results tell us? | When the crop is sensitive, but not when it's harvested. So the timing of the sensitive period was added to the tuning |
| 5 | What does a realistic harvest date cost in accuracy? | 13% for Finnish pea, about 5% for carrot and spinach, nothing for the rest. This led to the 5% rule |
| 6 | Could ERP delivery dates stand in for harvest dates? | No. Pea deliveries move 3 days between seasons while crop development moves 12, turnip rape alternates between August and September, and farms don't deliver in a consistent order. They're used only as a range |
| 7 | Do the faster versions of the code give the same results? | Yes. The full tuning run now takes about 25 seconds |

Run this after `c1_clean.ipynb` and `c1_weather_features.ipynb`. Experiment 5 reads the tuning results that `c1_weather_features.ipynb` saves.

## Setup

In [1]:
from pathlib import Path
import os, time
from dataclasses import replace
import numpy as np
import pandas as pd
from navigator.analysis import cropweather as cw
from navigator.analysis import yieldcv as cv

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
PKG = Path(os.environ.get("CCN_PKG", "../APETIT_CLIMATE_NATURE_NAVIGATOR_2026")).resolve()
OUT, RESULTS = Path("outputs"), Path("results")
assert (OUT / "c1_training_table.csv").exists() and (RESULTS / "calendar_tuning.csv").exists(), \
    "run c1_clean.ipynb and c1_weather_features.ipynb first"

keys, arr = cw.load_weather(PKG / "data/04_climate/weather_daily_1991_2060.parquet")
stress, sat = cw.soil_water(arr)
train = pd.read_csv(OUT / "c1_training_table.csv")
pyc = pd.read_csv(OUT / "parcel_year_contract.csv")
declared = pyc[pyc["status"].eq("declared")].assign(scenario="historical")
PREFIX = {"FI_SW": "GF", "SE_SKANE": "GS"}
hist = keys["scenario"].eq("historical").values

def hist_idx(regions):
    return np.where(hist & keys["cell_id"].str[:2].isin([PREFIX[r] for r in regions]).values)[0]

AGG = {}
def table(regions, crop, p, gap="rounded"):
    """Farm-year model table for one or more regions. gap='raw' reproduces the pre-fix immature_gap."""
    regions = tuple(regions)
    idx = hist_idx(regions)
    f = cw.crop_features(keys, arr, stress, sat, p, idx)
    if gap == "raw":   # rows are soil-major, then series in idx order
        f["immature_gap"] = np.tile(1.0 - cw.phenology(arr["tmean_c"][idx], p)["gdd_ratio"], len(cw.SOILS))
    if (regions, crop) not in AGG:
        AGG[(regions, crop)] = cw.FarmAggregator(keys, idx, declared[declared["crop"].eq(crop)])
    fy = AGG[(regions, crop)](f, cw.WEATHER_FEATURES)
    d = train[train["crop"].eq(crop) & train["region"].isin(regions)].merge(fy, on=["farm_id", "crop", "year"], how="left")
    return cv.add_model_columns(d)

### Older versions of the model fit

The model fit went through three versions. `v1` and `v2` are kept only to reproduce experiments 1 and 2. The slower pandas version of the cross-validation is kept for experiment 7.

In [2]:
def legacy_fit(X, y, mode, X_raw=None):
    """v1: ridge with an absolute penalty (lam=1) and no support check (first version).
       v2: penalty scaled by n (0.01 n) and features with < 5 % non-zero rows dropped,
           but support measured on the farm-demeaned values (second version).
       v3: as v2, support measured on the raw values (current yieldcv._fit)."""
    sd = X.std(0)
    if mode == "v1":
        sd = np.where(sd == 0, 1.0, sd)
        Xs = X / sd
        return np.linalg.solve(Xs.T @ Xs + 1.0 * np.eye(X.shape[1]), Xs.T @ y) / sd
    if mode == "v3":
        return cv._fit(X, y, 0.01, X_raw=X_raw)
    support = (np.abs(X) > 1e-9).mean(0)
    use = (support >= 0.05) & (sd > 1e-9)
    beta = np.zeros(X.shape[1])
    Xs = X[:, use] / sd[use]
    beta[use] = np.linalg.solve(Xs.T @ Xs + 0.01 * len(y) * np.eye(use.sum()), Xs.T @ y) / sd[use]
    return beta

def legacy_loyo(df, cols, mode="v3", min_train_years=3):
    """The original pandas implementation of leave-one-year-out CV."""
    df = df.dropna(subset=list(cols) + ["yield_t_ha"]).reset_index(drop=True)
    y = np.log(df["yield_t_ha"].values); X = df[list(cols)].values.astype(float); farm = df["farm_id"].values
    pred, base = np.full(len(df), np.nan), np.full(len(df), np.nan)
    for yr in np.unique(df["year"]):
        tr, te = df["year"].values != yr, df["year"].values == yr
        g = pd.DataFrame(X[tr]).groupby(farm[tr])
        xm = g.transform("mean").values
        ym = pd.Series(y[tr]).groupby(farm[tr]).transform("mean").values
        keep = pd.Series(farm[tr]).map(pd.Series(farm[tr]).value_counts()).values >= min_train_years
        beta = legacy_fit((X[tr] - xm)[keep], (y[tr] - ym)[keep], mode, X_raw=X[tr][keep])
        fm_y, fm_x = pd.Series(y[tr]).groupby(farm[tr]).mean(), g.mean()
        te_idx = np.where(te & pd.Series(farm).isin(fm_y.index).values)[0]
        base[te_idx] = fm_y.reindex(farm[te_idx]).values
        pred[te_idx] = base[te_idx] + (X[te_idx] - fm_x.reindex(farm[te_idx]).values) @ beta
    ok = ~np.isnan(pred)
    return {"n": int(ok.sum()), "rmse_log": float(np.sqrt(np.mean((y[ok] - pred[ok]) ** 2))),
            "rmse_log_baseline": float(np.sqrt(np.mean((y[ok] - base[ok]) ** 2)))}

## 1. First tuning run

We tried each combination of degree-days to harvest and sowing temperature, keeping all other settings at their starting values. Peas from both regions were scored together, and the drought and heat periods were fixed at their default share of the season.

This uses the code as it was then: the `v1` fit, and the failure-to-mature measure (`immature_gap`) before rounding.

In [3]:
GRID1 = {"pea": (range(600, 1301, 50), [6, 8, 10]), "turnip_rape": (range(900, 1501, 50), [6, 8, 10]),
         "potato": (range(900, 1701, 100), [6, 8, 10]), "carrot": (range(900, 1701, 100), [6, 8, 10]),
         "spinach": (range(300, 751, 50), [4, 6, 8])}
t0, rows = time.time(), []
for crop, (Ms, Ts) in GRID1.items():
    regions = ["FI_SW", "SE_SKANE"] if crop == "pea" else ["FI_SW"]
    for M in Ms:
        for T in Ts:
            p = replace(cw.DEFAULT_PARAMS[crop], gdd_maturity=float(M), sow_temp_c=float(T))
            rows.append({"crop": crop, "gdd_maturity": M, "sow_temp_c": T,
                         **legacy_loyo(table(regions, crop, p, gap="raw"), cv.model_features(crop), mode="v1")})
exp1 = pd.DataFrame(rows)
print(f"{len(exp1)} settings in {time.time() - t0:.0f}s")
for crop in GRID1:
    print(f"\n{crop}  (rmse of log yield; baseline {exp1.query('crop == @crop')['rmse_log_baseline'].iat[0]:.4f})")
    print(exp1.query("crop == @crop").pivot(index="sow_temp_c", columns="gdd_maturity", values="rmse_log").round(3).to_string())

168 settings in 9s

pea  (rmse of log yield; baseline 0.2024)
gdd_maturity   600    650   700    750    800    850    900    950    1000   1050   1100   1150   1200   1250   1300
sow_temp_c                                                                                                          
6             0.148  0.149  0.15  0.148  0.148  0.148  0.148  0.151  0.148  0.145  0.142  0.142  0.148  0.158  0.167
8             0.147  0.149  0.15  0.147  0.147  0.148  0.147  0.150  0.147  0.145  0.141  0.146  0.147  0.156  0.165
10            0.145  0.150  0.15  0.147  0.147  0.148  0.151  0.152  0.151  0.149  0.145  0.145  0.150  0.157  0.164

turnip_rape  (rmse of log yield; baseline 0.2258)
gdd_maturity   900    950    1000   1050   1100   1150   1200   1250   1300   1350   1400   1450   1500
sow_temp_c                                                                                             
6             0.150  0.137  0.132  0.129  0.128  0.127  0.126  0.132  0.140  0.140  0.144  0.1

Turnip rape (about 1,200 degree-days) and potato (about 1,400) have clear best values, and the sowing temperature hardly matters. For peas the error barely changes, staying between about 0.141 and 0.152 up to 1,200 degree-days, with a small dip at 1,100–1,150.

Spinach at 650 degree-days and carrot at 1,300 with a 10 °C sowing temperature give errors several times worse than the baseline. The settings next to them are fine, so this looked like a bug in the code.

Next we found the cause of those errors before trusting any of the results (experiment 2), and checked whether scoring the two pea regions together was hiding two different answers (experiment 3).

## 2. Why did spinach and carrot go wrong?

We took the worst case, spinach at 650 degree-days with a 4 °C sowing temperature, and looked at the error for each left-out season. Then we looked at the coefficients fitted when the worst season was left out.

In [4]:
p_bad = replace(cw.DEFAULT_PARAMS["spinach"], gdd_maturity=650.0, sow_temp_c=4.0)
d_raw = table(["FI_SW"], "spinach", p_bad, gap="raw")
cols = cv.model_features("spinach")

# Per-season error under the code of the time (v2 fit, unrounded gap)
dd = d_raw.dropna(subset=cols + ["yield_t_ha"]).reset_index(drop=True)
y = np.log(dd["yield_t_ha"].values); X = dd[cols].values.astype(float); farm = dd["farm_id"].values
per_year = {}
for yr in sorted(dd["year"].unique()):
    tr, te = dd["year"].values != yr, dd["year"].values == yr
    g = pd.DataFrame(X[tr]).groupby(farm[tr])
    xm = g.transform("mean").values; ym = pd.Series(y[tr]).groupby(farm[tr]).transform("mean").values
    beta = legacy_fit(X[tr] - xm, y[tr] - ym, "v2")
    fm = pd.Series(y[tr]).groupby(farm[tr]).mean()
    i = np.where(te & pd.Series(farm).isin(fm.index).values)[0]
    pred = fm.reindex(farm[i]).values + (X[i] - g.mean().reindex(farm[i]).values) @ beta
    per_year[yr] = float(np.sqrt(np.mean((y[i] - pred) ** 2)))
print("held-out season error:", {k: round(v, 2) for k, v in per_year.items()})
worst = max(per_year, key=per_year.get)

tr = dd["year"].values != worst
g = pd.DataFrame(X[tr]).groupby(farm[tr])
Xd = X[tr] - g.transform("mean").values
yd = y[tr] - pd.Series(y[tr]).groupby(farm[tr]).transform("mean").values
diag = pd.DataFrame({"non-zero share (demeaned)": (np.abs(Xd) > 1e-9).mean(0),
                     "non-zero share (raw)": (np.abs(X[tr]) > 1e-9).mean(0),
                     "sd (demeaned)": Xd.std(0), "beta (v2)": legacy_fit(Xd, yd, "v2")}, index=cols)
print(f"\nworst season: {worst}; training on the other seasons:")
diag.round(5)

held-out season error: {np.int64(2012): 0.11, np.int64(2013): 0.1, np.int64(2014): 0.11, np.int64(2015): 0.08, np.int64(2016): 0.09, np.int64(2017): 2.82, np.int64(2018): 0.08, np.int64(2019): 0.1, np.int64(2020): 0.16, np.int64(2021): 0.12, np.int64(2022): 0.12, np.int64(2023): 0.22, np.int64(2024): 0.09, np.int64(2025): 0.1, np.int64(2026): 0.1}

worst season: 2017; training on the other seasons:


,non-zero share (demeaned),non-zero share (raw),sd (demeaned),beta (v2)
drought_stage,1.00000,1.00000,0.17150,-0.25105
drought_x_irr,0.43662,0.43662,0.10849,0.19423
drought_early,1.00000,0.93662,0.06386,-0.33677
heat_days_27,0.97887,0.37324,1.28421,-0.01529
waterlog_clay,0.64789,0.09859,0.41404,-0.00725
harvest_rain_14_mm,1.00000,1.00000,20.44106,-0.00100
immature_gap,0.11972,0.01408,0.00027,-51.78187
year_c,0.97887,0.90141,4.24833,-0.00538
n_mineral_100,0.95070,0.95070,0.10240,0.10133


In [5]:
print("immature_gap by season (raw, mean over farms):")
print(d_raw.groupby("year")["immature_gap"].mean().map(lambda v: f"{v:.5f}").to_dict())

immature_gap by season (raw, mean over farms):
{2012: '0.00038', 2013: '0.00004', 2014: '0.00000', 2015: '0.00000', 2016: '0.00000', 2017: '0.05270', 2018: '0.00000', 2019: '0.00000', 2020: '0.00000', 2021: '0.00000', 2022: '0.00000', 2023: '0.00000', 2024: '0.00000', 2025: '0.00000', 2026: '0.00000'}


The failure-to-mature measure (`immature_gap`) should be zero in every season except 2017. In a few other seasons, rounding in the degree-day sum left tiny values of about 0.0003. In the raw data the measure is non-zero in only about 1% of rows. After subtracting each farm's average, those tiny values made it non-zero in about 12%.

The `v2` fit dropped measures that are rarely non-zero, but it checked after that subtraction, so it kept this one. It then fitted a coefficient of about −50 to what was only rounding noise. When 2017 was the left-out season, that coefficient met 2017's real value (about 0.05) and predicted a yield roughly 15 times too low.

We fixed it in two places. `cropweather.crop_features` now sets values below 0.001 to exactly zero, and `yieldcv._fit` checks how often a measure is non-zero on the raw data. The table compares the same settings under all three versions of the code:

In [6]:
cases = [("spinach", 650, 4), ("carrot", 1300, 10), ("spinach", 450, 8)]
rows = []
for crop, M, T in cases:
    p = replace(cw.DEFAULT_PARAMS[crop], gdd_maturity=float(M), sow_temp_c=float(T))
    raw, fixed = table(["FI_SW"], crop, p, gap="raw"), table(["FI_SW"], crop, p)
    c = cv.model_features(crop)
    rows.append({"setting": f"{crop} {M} / {T} °C",
                 "v1 fit, raw gap": legacy_loyo(raw, c, "v1")["rmse_log"],
                 "v2 fit, raw gap": legacy_loyo(raw, c, "v2")["rmse_log"],
                 "current code": cv.loyo(fixed, c)[0]["rmse_log"],
                 "baseline": cv.loyo(fixed, c)[0]["rmse_log_baseline"]})
pd.DataFrame(rows).set_index("setting").round(4)

,"v1 fit, raw gap","v2 fit, raw gap",current code,baseline
setting,,,,
spinach 650 / 4 °C,0.7351,0.7320,0.1196,0.1224
carrot 1300 / 10 °C,1.0948,1.0969,0.1177,0.1432
spinach 450 / 8 °C,0.1058,0.1058,0.1057,0.1224


With the current code the wrong results are gone, and a setting that was already fine (spinach at 450 degree-days, 8 °C) gives the same result as before. All later experiments use the current code.

## 3. Can Finnish and Swedish peas share one calendar?

Scoring both regions together assumes each farm's average absorbs the difference in yield level between them. But the two regions could be sensitive at different points in the season. We ran the same settings (8 °C sowing temperature, default sensitive periods) on both regions together and on each one separately.

In [7]:
rows = []
for M in range(600, 1301, 50):
    p = replace(cw.DEFAULT_PARAMS["pea"], gdd_maturity=float(M), sow_temp_c=8.0)
    for name, regions in [("pooled", ["FI_SW", "SE_SKANE"]), ("FI_SW", ["FI_SW"]), ("SE_SKANE", ["SE_SKANE"])]:
        rows.append({"fit": name, "gdd_maturity": M, **cv.loyo(table(regions, "pea", p), cv.model_features("pea"))[0]})
exp3 = pd.DataFrame(rows)
print(exp3.loc[exp3.groupby("fit")["rmse_log"].idxmin(), ["fit", "gdd_maturity", "rmse_log", "improvement_pct"]].round(4).to_string(index=False))
exp3.pivot(index="gdd_maturity", columns="fit", values="rmse_log").round(4).T

     fit  gdd_maturity  rmse_log  improvement_pct
   FI_SW          1150    0.1306          44.2752
SE_SKANE           600    0.1278          32.1623
  pooled          1150    0.1413          30.1967


gdd_maturity,600,650,700,750,800,850,900,950,1000,1050,1100,1150,1200,1250,1300
fit,,,,,,,,,,,,,,,
FI_SW,0.1925,0.1782,0.1681,0.1670,0.1650,0.1548,0.1553,0.1478,0.1421,0.1459,0.1411,0.1306,0.1330,0.1357,0.1360
SE_SKANE,0.1278,0.1385,0.1408,0.1376,0.1360,0.1383,0.1405,0.1479,0.1525,0.1512,0.1485,0.1525,0.1581,0.1703,0.1708
pooled,0.1465,0.1488,0.1495,0.1472,0.1469,0.1479,0.1468,0.1504,0.1469,0.1451,0.1416,0.1413,0.1469,0.1569,0.1652


Finland's error falls steadily to its lowest point at about 1,150 degree-days. Sweden's is lowest at the bottom of the range, about 600, and rises from there. Scoring them together averages two different curves, which is why the pea results looked flat in experiment 1.

So Finnish and Swedish peas are now tuned separately.

## 4. What do the pea results tell us?

Finland's best setting, about 1,150 degree-days, puts the pea harvest in mid-August, but peas are delivered in July. Is the tuning finding the harvest date, or something else?

For each region, we added two more settings to try: when the drought-sensitive period starts, as a share of the season, and how long it lasts. The heat-sensitive period starts at the same time and lasts a quarter of the season.

In [8]:
t0, rows = time.time(), []
for region in ["FI_SW", "SE_SKANE"]:
    for M in range(500, 1101, 100):
        for lo in [0.20, 0.35, 0.50, 0.65]:
            for span in [0.30, 0.50]:
                for T in [6, 8]:
                    p = replace(cw.DEFAULT_PARAMS["pea"], gdd_maturity=float(M), sow_temp_c=float(T),
                                sensitive=(lo, round(lo + span, 2)), flower=(lo, round(min(lo + 0.25, 1.0), 2)))
                    rows.append({"region": region, "gdd_maturity": M, "win_start": lo, "win_span": span, "sow_temp_c": T,
                                 "window_gdd": f"{lo * M:.0f}-{min(lo + span, 1) * M:.0f}",
                                 **cv.loyo(table([region], "pea", p), cv.model_features("pea"))[0]})
exp4 = pd.DataFrame(rows)
print(f"{len(exp4)} settings in {time.time() - t0:.0f}s")
for region, g in exp4.groupby("region"):
    print(f"\n{region}: best settings")
    print(g.nsmallest(6, "rmse_log")[["gdd_maturity", "win_start", "win_span", "sow_temp_c", "window_gdd", "rmse_log", "improvement_pct"]]
          .round(4).to_string(index=False))

224 settings in 7s

FI_SW: best settings
 gdd_maturity  win_start  win_span  sow_temp_c window_gdd  rmse_log  improvement_pct
         1100       0.50       0.3           6    550-880    0.1177          49.7964
         1100       0.50       0.3           8    550-880    0.1210          48.3652
          900       0.65       0.3           6    585-855    0.1211          48.3363
          900       0.65       0.5           6    585-900    0.1218          48.0565
         1100       0.50       0.5           6   550-1100    0.1232          47.4515
          900       0.65       0.3           8    585-855    0.1253          46.5338

SE_SKANE: best settings
 gdd_maturity  win_start  win_span  sow_temp_c window_gdd  rmse_log  improvement_pct
          600       0.35       0.5           8    210-510    0.1277          32.2546
          600       0.50       0.5           8    300-600    0.1286          31.7466
          600       0.50       0.3           8    300-480    0.1288          31.6387

Finland: the lowest error for each combination of degree-days to harvest and start of the sensitive period.

In [9]:
exp4.query("region == 'FI_SW'").groupby(["gdd_maturity", "win_start"])["rmse_log"].min().unstack().round(4)

win_start,0.20,0.35,0.50,0.65
gdd_maturity,,,,
500,0.2097,0.2130,0.2130,0.2087
600,0.2021,0.1990,0.1903,0.1878
700,0.1874,0.1746,0.1680,0.1570
800,0.1881,0.1697,0.1587,0.1335
900,0.2040,0.1721,0.1429,0.1211
1000,0.1991,0.1613,0.1339,0.1350
1100,0.1827,0.1404,0.1177,0.1614


In Finland the two best settings are:

- a long season (1,100 degree-days) with the sensitive period starting halfway through;
- a shorter season (900) with it starting at 65%.

Both put the sensitive period at roughly the same point, about 550–880 degree-days after sowing. The table shows the same pattern: the error is low wherever the sensitive period starts near 550–600 degree-days, whatever the season length. So the yields tell us when the crop is vulnerable. They don't tell us when it's harvested.

Sweden prefers a short season, with the sensitive period in the first 600 or so degree-days. Its exact start shifts with the sowing temperature, and the final tuning settles on 390–600.

As a result, the timing of the sensitive period is now tuned too, and the harvest date is checked against the ERP delivery dates (experiment 5). Letting the sensitive period start later also raised Finnish pea from about 44% to about 50% better than the baseline.

## 5. What does a realistic harvest date cost?

For each region and crop: how much worse is the best calendar that's ready inside the delivery range, compared with the best calendar overall? This reads the full tuning results in `results/calendar_tuning.csv` (848 settings).

In [10]:
tun = pd.read_csv(RESULTS / "calendar_tuning.csv")
to_date = lambda d: (pd.Timestamp(2001, 1, 1) + pd.Timedelta(days=float(d) - 1)).strftime("%d %b") if pd.notna(d) else "never"
rows = []
for (region, crop), g in tun.groupby(["region", "crop"], sort=False):
    b = g.loc[g["rmse_log"].idxmin()]
    pl = g[g["ready_plausible"]]
    bp = pl.loc[pl["rmse_log"].idxmin()]
    rows.append({"region": region, "crop": crop, "settings": len(g), "plausible settings": len(pl),
                 "best: window": b["window_gdd"], "best: ready": to_date(b["ready_doy_median"]), "best: rmse": b["rmse_log"],
                 "plausible: window": bp["window_gdd"], "plausible: ready": to_date(bp["ready_doy_median"]),
                 "plausible: rmse": bp["rmse_log"], "cost of plausibility %": 100 * (bp["rmse_log"] / b["rmse_log"] - 1)})
pd.DataFrame(rows).set_index(["region", "crop"]).round(4)

settings  plausible settings best: window best: ready  best: rmse plausible: window plausible: ready  plausible: rmse  cost of plausibility %
region   crop                                                                                                                                                      
FI_SW    pea               128                  64      550-880      14 Aug      0.1177           520-760           22 Jul           0.1335                 13.4211
SE_SKANE pea               128                  80      390-600      04 Jul      0.1285           390-600           04 Jul           0.1285                  0.0000
FI_SW    turnip_rape       112                 112      455-845      02 Sep      0.1188           455-845           02 Sep           0.1188                  0.0000
         potato            144                  96     490-1190      12 Sep      0.0844          490-1190           12 Sep           0.0844                  0.0000
         carrot            176                 112      455-700      16 Jul      0.1053           240-840           25 Aug           0.1102                  4.6832
         spinach           160                  64      262-487      17 Jul      0.1052           228-350           15 Jun           0.1101                  4.7234

Swedish pea, turnip rape and potato are already ready on time at their best setting. Carrot and spinach lose just under 5%. Finnish pea loses about 13%.

So the pipeline takes a calendar that's ready on time if it costs at most 5%. The delivery dates mostly follow the factory's schedule (experiment 6). That makes them good for settling close calls and for crops with few farms (carrot and spinach have 12–16 each). They shouldn't overrule a clear signal from 1,043 Finnish pea yield records, so Finnish pea keeps its later calendar and is flagged.

## 6. Could ERP delivery dates stand in for harvest dates?

The ERP records the date each farm delivered each contract crop in 2021–25. If those dates moved with the weather, they would tell us when crops were really ready, and we could tune against them.

We checked two things. First, whether the typical delivery date in each season tracks how warm that season was (degree-days from 1 April to 31 July, averaged over the Finnish cells). Second, whether a farm that delivers early one year also delivers early the next.

In [11]:
erp = pd.read_csv(PKG / "data/02_raw_feeds/erp_purchase_lines_2021_2025.csv", parse_dates=["posting_date"])
MAT2CROP = {"M110": "pea", "M100": "turnip_rape", "M111": "potato", "M112": "carrot", "M113": "spinach"}
dl = erp[erp["material_id"].isin(list(MAT2CROP)) & erp["movement_type"].eq(101)].copy()
dl["crop"], dl["year"], dl["doy"] = dl["material_id"].map(MAT2CROP), dl["posting_date"].dt.year, dl["posting_date"].dt.dayofyear

fi = np.where(hist & keys["cell_id"].str.startswith("GF").values)[0]
t_jul = cw.t_of("07-31"); t_apr = cw.t_of("04-01")
gdd = np.clip(arr["tmean_c"][fi, t_apr:t_jul + 1] - 5, 0, None).sum(1)
warmth = pd.Series(gdd, index=keys["year"].values[fi]).groupby(level=0).mean().rename("gdd_apr_jul")

med = dl.groupby(["year", "crop"])["doy"].median().unstack()
print("(a) median delivery day-of-year vs season warmth")
print(med.join(warmth).round(0).to_string())
print("\ncorrelation with warmth (5 seasons):", med.corrwith(warmth.reindex(med.index)).round(2).to_dict())
print("range of the median across seasons (days):", (med.max() - med.min()).to_dict())

# For comparison: how far the degree-day calendar moves a fixed target (900 degree-days from 20 Apr) in the same seasons
sow = cw.t_of("04-20")
cg = np.cumsum(np.clip(arr["tmean_c"][fi, sow:] - 5, 0, None), axis=1)
ready = pd.Series(sow + (cg >= 900).argmax(1) + 60, index=keys["year"].values[fi]).groupby(level=0).median()
print("900 degree-days reached (day of year), 2021-25:", ready.loc[2021:2025].to_dict(),
      "-> range", int(ready.loc[2021:2025].max() - ready.loc[2021:2025].min()), "days")

(a) median delivery day-of-year vs season warmth
      carrot    pea  potato  spinach  turnip_rape  gdd_apr_jul
year                                                          
2021   256.0  195.0   249.0    166.0        229.0       1100.0
2022   258.0  194.0   258.0    168.0        256.0       1085.0
2023   253.0  197.0   260.0    165.0        227.0       1080.0
2024   260.0  197.0   261.0    168.0        258.0        950.0
2025   256.0  195.0   253.0    164.0        226.0       1133.0

correlation with warmth (5 seasons): {'carrot': -0.69, 'pea': -0.62, 'potato': -0.66, 'spinach': -0.6, 'turnip_rape': -0.72}
range of the median across seasons (days): {'carrot': 7.5, 'pea': 3.0, 'potato': 12.0, 'spinach': 3.5, 'turnip_rape': 32.0}
900 degree-days reached (day of year), 2021-25: {2021: 200.0, 2022: 201.0, 2023: 200.0, 2024: 211.0, 2025: 199.0} -> range 12 days


In [12]:
def farm_persistence(c):
    m = dl[dl["crop"].eq(c)].groupby(["supplier_id", "year"])["doy"].mean().unstack()
    r = m.corr().values
    return r[np.triu_indices_from(r, 1)].mean()
print("(b) mean year-to-year correlation of a farm's delivery date:",
      {c: round(farm_persistence(c), 2) for c in ["pea", "turnip_rape"]})

(b) mean year-to-year correlation of a farm's delivery date: {'pea': np.float64(-0.0), 'turnip_rape': np.float64(-0.0)}


Deliveries do come a little later in the one cool season, 2024, so they correlate with warmth. But five seasons is very little evidence, and the shifts are small. The typical pea delivery date moves by 3 days, while a fixed degree-day target moves by 12 over the same seasons, and by 2–3 weeks across 2012–26.

Turnip rape arrives in August in 2021, 2023 and 2025 and in September in 2022 and 2024, and the warm 2022 season is one of the late ones. A farm's place in the queue changes randomly from year to year (correlation about zero). The dates look like a factory schedule that shifts slightly with the season.

So delivery dates aren't used as harvest dates. The calendar is tuned on yields, and the delivery dates only set a range (experiment 5).

## 7. Do the faster versions give the same results?

The full tuning run (848 settings) was too slow with the original pandas code for averaging parcels into farms and for the cross-validation. We rewrote both in numpy (`cropweather.FarmAggregator` and `yieldcv.loyo`). Does any number change?

In [13]:
rows = []
for region, crop in [("FI_SW", "pea"), ("SE_SKANE", "pea"), ("FI_SW", "potato"), ("FI_SW", "turnip_rape")]:
    p = cw.DEFAULT_PARAMS[crop]
    idx = hist_idx([region])
    f = cw.crop_features(keys, arr, stress, sat, p, idx)
    dc = declared[declared["crop"].eq(crop)]
    t0 = time.perf_counter(); slow = cw.to_farms(cw.to_parcels(f, dc), cw.WEATHER_FEATURES).dropna(subset=["season_gdd"]); t_slow = time.perf_counter() - t0
    agg = cw.FarmAggregator(keys, idx, dc)
    t0 = time.perf_counter(); fast = agg(f, cw.WEATHER_FEATURES); t_fast = time.perf_counter() - t0
    m = slow.merge(fast, on=["farm_id", "crop", "year"], suffixes=("_slow", "_fast"))
    max_diff = max(np.nanmax(np.abs(m[c + "_slow"] - m[c + "_fast"])) for c in cw.WEATHER_FEATURES)
    d = table([region], crop, p)
    t0 = time.perf_counter(); a = legacy_loyo(d, cv.model_features(crop)); t_cv_slow = time.perf_counter() - t0
    t0 = time.perf_counter(); b = cv.loyo(d, cv.model_features(crop))[0]; t_cv_fast = time.perf_counter() - t0
    rows.append({"unit": f"{region}/{crop}", "farm-years": len(slow), "matched": len(m), "max feature diff": max_diff,
                 "rmse pandas": a["rmse_log"], "rmse numpy": b["rmse_log"],
                 "aggregate ms (slow / fast)": f"{1e3 * t_slow:.0f} / {1e3 * t_fast:.1f}",
                 "CV ms (slow / fast)": f"{1e3 * t_cv_slow:.0f} / {1e3 * t_cv_fast:.1f}"})
exp7 = pd.DataFrame(rows).set_index("unit")
# equal up to float summation order
assert (exp7["max feature diff"] < 1e-9).all() and np.allclose(exp7["rmse pandas"], exp7["rmse numpy"], rtol=0, atol=1e-10)
exp7.round(6)

,farm-years,matched,max feature diff,rmse pandas,rmse numpy,aggregate ms (slow / fast),CV ms (slow / fast)
unit,,,,,,,
FI_SW/pea,1738,1738,0.0,0.154968,0.154968,21 / 2.4,21 / 5.5
SE_SKANE/pea,4460,4460,0.0,0.140053,0.140053,20 / 3.7,18 / 7.0
FI_SW/potato,222,222,0.0,0.087627,0.087627,7 / 1.5,14 / 2.2
FI_SW/turnip_rape,3387,3387,0.0,0.123552,0.123552,16 / 3.4,26 / 8.7


No. The measures and the errors match the old code, with any differences at the level of rounding, and both steps run several times faster. The full tuning run now takes about 25 seconds.